In [1]:
import langchain_core, langchain_community, langchain_chroma, langchain_classic
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt, dotenv
dotenv.load_dotenv()


c:\Users\Neel\Desktop\Azure_open_ai\myenv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


True

In [2]:
from langchain_openai import *
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

c:\Users\Neel\Desktop\Azure_open_ai\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownTextSplitter, MarkdownHeaderTextSplitter
import pymupdf4llm 
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader, ToMarkdownLoader

In [4]:
pdf_path =r"C:\Users\Neel\Desktop\Azure_open_ai\Loader\pdfs\Must-Read on AI Agents.pdf"

In [6]:
pdf_path 

'C:\\Users\\Neel\\Desktop\\Azure_open_ai\\Loader\\pdfs\\pandas_datetime_book.pdf'

In [5]:
import tempfile
md_doc = pymupdf4llm.to_markdown(pdf_path, page_chunks=True)

from langchain_core.documents import Document

docs=[Document(page_content=doc['text'], metadata = doc['metadata']) for doc in md_doc]

spliters = MarkdownHeaderTextSplitter(
  headers_to_split_on=[('#', "Header_1"),
                       ('##', "Header_2"),
                       ('###', "Header_3")]
  
)

spliter_recursive = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=50)

import re
def clean_text(text):
  # Page numbers remove karo
  text = re.sub(r'^\d+$', '', text, flags=re.MULTILINE)
  # Image placeholders remove karo
  text = re.sub(r'==>.*?<==', '', text)
  # Extra whitespace remove karo
  text = re.sub(r'\n{3,}', '\n\n', text)
  return text.strip()


all_chunk =[]
for i,doc in enumerate(docs):
  clean = clean_text(doc.page_content)
  chunks = spliter_recursive.split_text(clean)
  documents = [ Document(page_content=chunk, metadat = doc.metadata) for chunk in chunks]
  all_chunk.extend(documents)
all_chunk
  



[Document(metadata={}, page_content='# A practical guide to building agents \n\n****'),
 Document(metadata={}, page_content='## Contents \n\n****\n\n**----- Start of picture text -----**<br>\nWhat is an agent? 4<br>When should you build an agent? 5<br>Agent design foundations 7<br>Guardrails 24<br>Conclusion 32<br>**----- End of picture text -----**<br>\n\nPractical guide to building agents \n\n2'),
 Document(metadata={}, page_content='## Introduction \n\nLarge language models are becoming increasingly capable of handling complex, multi-step tasks. Advances in reasoning, multimodality, and tool use have unlocked a new category of LLM-powered systems known as agents.'),
 Document(metadata={}, page_content='This guide is designed for product and engineering teams exploring how to build their first agents, distilling insights from numerous customer deployments into practical and actionable best practices. It includes frameworks for identifying promising use cases, clear patterns for desig

In [6]:
docs[0]

Document(metadata={'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'creationDate': 'D:20250407142051Z', 'modDate': 'D:20250407142054Z', 'trapped': '', 'encryption': None, 'file_path': 'C:\\Users\\Neel\\Desktop\\Azure_open_ai\\Loader\\pdfs\\Must-Read on AI Agents.pdf', 'page_count': 34, 'page_number': 1}, page_content='# A practical guide to building agents \n\n**==> picture [542 x 349] intentionally omitted <==**\n\n')

In [ ]:
vectore_store = Chroma.from_documents(documents=all_chunk, 
                                      embedding=HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2',
                                                                      
                                                                      
                                                                    ),
                                                                    collection_name='my_chorma_collection3',
                                                                    persist_directory='chorma_db_new')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5610.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
base_ret=vectore_store.as_retriever(
  search_type =  'mmr',
  search_kwargs ={'k':3, 'lambda_mult':0.5}
)

In [9]:
res=base_ret.invoke('what is agent')

In [10]:
res

[Document(id='4327b81e-d9f8-4695-acfe-be2555e79d3f', metadata={}, page_content="## What is an agent? \n\nWhile conventional software enables users to streamline and automate workflows, agents are able to perform the same workflows on the users’ behalf with a high degree of independence. \n\nAgents are systems that independently accomplish tasks on your behalf. \n\nA workflow is a sequence of steps that must be executed to meet the user’s goal, whether that's resolving a customer service issue, booking a restaurant reservation, committing a code change, or generating a report."),
 Document(id='ccb3f7f8-2115-4fba-a23c-0d7c970a1082', metadata={}, page_content='## Agent design foundations \n\nIn its most fundamental form, an agent consists of three core components: \n\n01 Model The LLM powering the agent’s reasoning and decision-making 02 Tools External functions or APIs the agent can use to take action 03 Instructions Explicit guidelines and guardrails defining how the agent behaves \n\nH

In [11]:
for i in res:
  # print(i.metadata)
  # print("**********")
  print(i.page_content)


## What is an agent? 

While conventional software enables users to streamline and automate workflows, agents are able to perform the same workflows on the users’ behalf with a high degree of independence. 

Agents are systems that independently accomplish tasks on your behalf. 

A workflow is a sequence of steps that must be executed to meet the user’s goal, whether that's resolving a customer service issue, booking a restaurant reservation, committing a code change, or generating a report.
## Agent design foundations 

In its most fundamental form, an agent consists of three core components: 

01 Model The LLM powering the agent’s reasoning and decision-making 02 Tools External functions or APIs the agent can use to take action 03 Instructions Explicit guidelines and guardrails defining how the agent behaves 

Here’s what this looks like in code when using OpenAI’s Agents SDK. You can also implement the same concepts using your preferred library or building directly from scratch. 

#